# QLoRA lab · Qwen2.5-1.5B dialogue summarization

Compare the same Qwen checkpoint before and after adapter training on DialogSum. The graded comparison uses 2,000 seeded training rows before length filtering, 60 seeded validation rows, 4-bit NF4, and greedy decoding. The test split is held out.

**Run status.** This workspace has no CUDA GPU. Saved outputs show what was executed here. Cells that require the T4 print a skip status on a CPU. In Kaggle, enable Internet and a GPU T4 accelerator, then run all cells. Exercise 7 requires a separate rating step after the two sets of summaries are generated. The CPU diagnostic at the end uses a different 4-bit format and is labeled separately.

In [1]:
# Kaggle installation. If upgrading already-imported packages, restart the kernel.
# If PEFT reports an incompatible torchao version, uninstall torchao and restart.
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
ON_KAGGLE = Path("/kaggle/working").is_dir()
if ON_KAGGLE:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==5.0.0", "peft==0.21.0", "accelerate",
        "datasets", "bitsandbytes", "rouge_score",
    ])
else:
    print("Kaggle-only installation skipped.")

Kaggle-only installation skipped.


In [2]:
import math
import random
import time
from importlib.metadata import version

try:
    import torch
except ModuleNotFoundError:
    torch = None
from transformers import AutoTokenizer
from datasets import load_dataset
from rouge_score import rouge_scorer

SEED = 42
random.seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)
GB = 1e9
CKPT = "Qwen/Qwen2.5-1.5B-Instruct"
N_TRAIN, N_EVAL, MAX_LEN = 2000, 60, 512
OUT = Path("/kaggle/working/qlora_dialogsum" if ON_KAGGLE else "qlora_dialogsum_outputs")
ADAPTER_DIR = OUT / "adapter"
HAS_GPU = torch is not None and torch.cuda.is_available()

print("torch", torch.__version__ if torch is not None else "unavailable",
      "| transformers", version("transformers"))
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none")

torch 2.10.0+cpu | transformers 5.0.0
GPU: none


## Exercise 1 · Memory budget

In [3]:
N_TOTAL, N_LINEAR, N_OTHER, N_LORA = 1_543_714_304, 1_310_195_712, 233_518_592, 18_464_768
NF4_BYTES = (4 + 0.127) / 8

def memory_budget(n_total, n_linear, n_other, n_lora):
    return {
        "full FT": 16 * n_total / GB,
        "LoRA": (2 * n_total + 16 * n_lora) / GB,
        "QLoRA": (NF4_BYTES * n_linear + 4 * n_other + 16 * n_lora) / GB,
    }

estimate = memory_budget(N_TOTAL, N_LINEAR, N_OTHER, N_LORA)
for job, gb in estimate.items():
    print(f"{job:<8} {gb:6.2f} GB   fits on a 16 GB T4 before activations: {gb < 16}")
assert abs(estimate["full FT"] - 24.70) < 0.01
assert abs(estimate["LoRA"] - 3.38) < 0.01
assert abs(estimate["QLoRA"] - 1.91) < 0.01
print(f"Fivefold QLoRA estimate: {5 * estimate['QLoRA']:.2f} GB")
print("Exercise 1 checks passed.")

full FT   24.70 GB   fits on a 16 GB T4 before activations: False
LoRA       3.38 GB   fits on a 16 GB T4 before activations: True
QLoRA      1.91 GB   fits on a 16 GB T4 before activations: True
Fivefold QLoRA estimate: 9.53 GB
Exercise 1 checks passed.


**Question 1.** (a) Which of the three jobs fit on a T4 before activations? (b) For this 1.5B model, plain LoRA in 16-bit would also fit. Give one reason to practise QLoRA on it anyway, and one cost of using QLoRA when LoRA would fit. (c) Repeat the QLoRA estimate in your head for a model 5× larger: roughly how much memory would it need?

**(a)** LoRA (3.38 GB) and QLoRA (1.91 GB) fit before activations. Full fine-tuning (24.70 GB) exceeds 16 GB.  
**(b)** This small checkpoint is a practical way to learn a workflow that leaves more room for larger models or longer inputs. One cost is quantization error in the frozen base weights.  
**(c)** Roughly 9.53 GB before activations if the quantized, non-quantized, and adapter parameter counts all scale fivefold.

## Exercise 2 · Chat formatting and summary-only loss

In [4]:
local_data = os.environ.get("QLORA_LOCAL_DATA")
if local_data:
    paths = {split: str(Path(local_data) / f"{split}.csv") for split in ("train", "validation")}
    raw = load_dataset("csv", data_files=paths)
else:
    from datasets import DatasetDict
    train, validation = load_dataset("knkarthick/dialogsum", split=["train", "validation"])
    raw = DatasetDict(train=train, validation=validation)

tokenizer = AutoTokenizer.from_pretrained(os.environ.get("QLORA_LOCAL_TOKENIZER", CKPT))
example = raw["train"][0]
print(raw)
print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("\n--- dialogue ---\n" + example["dialogue"][:600])
print("\n--- reference summary ---\n" + example["summary"])

INSTRUCTION = (
    "Summarize the following dialogue in one or two sentences. "
    "Refer to the speakers as #Person1#, #Person2#, ... as they appear in the dialogue."
)
END = "<|im_end|>"

def make_prompt(dialogue):
    messages = [{"role": "user", "content": f"{INSTRUCTION}\n\n{dialogue}"}]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

print("\n--- chat prompt ---\n" + make_prompt("#Person1#: Hi.\n#Person2#: Hello!"))

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
})
pad token: <|endoftext|> 151643

--- dialogue ---
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1

--- reference summary ---
Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every y

In [5]:
def build_example(dialogue, summary):
    prompt_ids = tokenizer(make_prompt(dialogue), add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(summary + END, add_special_tokens=False)["input_ids"]
    input_ids = prompt_ids + answer_ids
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + answer_ids,
    }

ex = build_example(example["dialogue"], example["summary"])
n_prompt = len(tokenizer(make_prompt(example["dialogue"]), add_special_tokens=False)["input_ids"])
assert set(ex) == {"input_ids", "attention_mask", "labels"}
assert len(ex["input_ids"]) == len(ex["labels"]) == len(ex["attention_mask"])
assert all(label == -100 for label in ex["labels"][:n_prompt])
assert ex["labels"][n_prompt:] == ex["input_ids"][n_prompt:]
assert ex["input_ids"][-1] == tokenizer.convert_tokens_to_ids(END)
print(f"{len(ex['input_ids'])} tokens, of which {len(ex['input_ids']) - n_prompt} are scored")
print("scored text:", tokenizer.decode(ex["labels"][n_prompt:]))
print("Exercise 2 checks passed.")

347 tokens, of which 39 are scored
scored text: Mr. Smith's getting a check-up, and Doctor Hawkins advises him to have one every year. Hawkins'll give some information about their classes and medications to help Mr. Smith quit smoking.<|im_end|>
Exercise 2 checks passed.


In [6]:
train_rows = raw["train"].shuffle(seed=SEED).select(range(N_TRAIN))
examples = [build_example(row["dialogue"], row["summary"]) for row in train_rows]
lengths = [len(row["input_ids"]) for row in examples]
train_data = [row for row in examples if len(row["input_ids"]) <= MAX_LEN]
del examples
total_tokens = sum(len(row["input_ids"]) for row in train_data)
scored_tokens = sum(sum(label != -100 for label in row["labels"]) for row in train_data)
print(f"training examples: {len(train_data)} kept, {N_TRAIN - len(train_data)} dropped (> {MAX_LEN} tokens)")
print(f"tokens per example: median {sorted(lengths)[len(lengths) // 2]}, max {max(lengths)}")
print(f"scored tokens: {scored_tokens:,} of {total_tokens:,} ({scored_tokens / total_tokens:.1%})")

eval_rows = raw["validation"].shuffle(seed=SEED).select(range(N_EVAL))
eval_dialogues, eval_refs = list(eval_rows["dialogue"]), list(eval_rows["summary"])

training examples: 1940 kept, 60 dropped (> 512 tokens)
tokens per example: median 265, max 1231
scored tokens: 65,739 of 536,797 (12.2%)


**Question 2.** (a) What would the model learn if the prompt tokens were **not** masked? Why is that wasteful here? Use the printed numbers. (b) Why must the answer end with `<|im_end|>`? What would you expect at generation time if it did not? (c) Why drop over-long examples instead of truncating them to `MAX_LEN`?

**(a)** Unmasking the prompt would also train the model to predict the instruction, chat markers, and dialogue. Only 65,739 of 536,797 positions (12.2%) carry summary or stop targets. The remaining 471,058 positions would reward reproducing input text if unmasked. The prompt remains visible to attention when its labels are masked.  
**(b)** The end-of-turn token gives the model a supervised target for stopping. Without it, the pretrained model may still stop, but the adapter gets no direct signal to stop after its summary. It may continue with more text or hit the generation cap.  
**(c)** Right-truncating a prompt-plus-answer sequence can remove the summary tail, the stop token, or every supervised token. Filtering keeps complete targets, at the cost of excluding the longest dialogues.

## Exercise 3 · Load the NF4 model on GPU 0

In [7]:
expected_inference_gb = (NF4_BYTES * N_LINEAR + 2 * N_OTHER) / GB
print(f"4-bit inference weight estimate: {expected_inference_gb:.2f} GB")

if HAS_GPU:
    import bitsandbytes as bnb
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        CKPT, quantization_config=bnb_config, dtype=torch.float16, device_map={"": 0}
    )
    n4 = sum(isinstance(module, bnb.nn.Linear4bit) for module in model.modules())
    loaded_gb = torch.cuda.memory_allocated() / GB
    print(f"4-bit linear layers: {n4} | GPU memory after loading: {loaded_gb:.2f} GB")
    print("compute dtype:", bnb_config.bnb_4bit_compute_dtype,
          "| quant type:", bnb_config.bnb_4bit_quant_type)
    print(f"Measured minus estimate: {loaded_gb - expected_inference_gb:+.2f} GB")
    assert n4 == 28 * 7
    assert bnb_config.bnb_4bit_quant_type == "nf4" and bnb_config.bnb_4bit_use_double_quant
    assert loaded_gb < 2.0
    print("Exercise 3 checks passed.")
else:
    print("Exercise 3 model load and GPU measurement skipped: CUDA unavailable.")

4-bit inference weight estimate: 1.14 GB
Exercise 3 model load and GPU measurement skipped: CUDA unavailable.


**Question 3.** Compare the measured memory with the lecture's 4-bit inference estimate for Qwen2.5-1.5B (about 1.14 GB). What could explain a difference? Why did we choose `torch.float16` as the compute dtype, and what would you use on an A100?

The theoretical weights total about **1.14 GB**. The cell reports the measured difference on a T4. Quantization metadata, auxiliary tensors, padding, and actual tensor dtypes can move the measurement away from the parameter estimate. The measurement is PyTorch's allocated bytes at load time. Reserved memory and CUDA context overhead belong to different measurements.

FP16 uses the T4's FP16 hardware support. An A100 supports BF16, whose wider exponent range makes overflow less likely during mixed-precision training.

## Exercise 4 · Original model and ROUGE

In [8]:
ROUGE_KEYS = ("rouge1", "rouge2", "rougeL")
scorer = rouge_scorer.RougeScorer(ROUGE_KEYS, use_stemmer=True)

def rouge(preds, refs):
    if len(preds) != len(refs) or not refs:
        raise ValueError("ROUGE needs equally sized, nonempty predictions and references.")
    scores = [scorer.score(ref, pred) for ref, pred in zip(refs, preds)]
    return {
        key: 100 * sum(pair[key].fmeasure for pair in scores) / len(scores)
        for key in ROUGE_KEYS
    }

def words(texts):
    return sum(len(text.split()) for text in texts) / len(texts)

assert abs(rouge(["the cat sat"], ["the cat sat"])["rouge1"] - 100) < 1e-6
assert rouge(["a dog ran"], ["the cat sat"])["rouge2"] == 0
print("Exercise 4 ROUGE function checks passed.")

Exercise 4 ROUGE function checks passed.


In [9]:
def generate_summaries(model, dialogues, batch_size=8, max_new_tokens=96):
    model.eval()
    tokenizer.padding_side = "left"
    outputs = []
    with torch.no_grad():
        for start in range(0, len(dialogues), batch_size):
            prompts = [make_prompt(d) for d in dialogues[start:start + batch_size]]
            enc = tokenizer(
                prompts, return_tensors="pt", padding=True, add_special_tokens=False,
            ).to(model.device)
            gen = model.generate(
                **enc, max_new_tokens=max_new_tokens, do_sample=False,
                temperature=None, top_p=None, top_k=None, pad_token_id=tokenizer.pad_token_id,
            )
            outputs.extend(tokenizer.batch_decode(
                gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True
            ))
    return [output.strip() for output in outputs]

if HAS_GPU:
    start = time.perf_counter()
    base_preds = generate_summaries(model, eval_dialogues)
    print(f"Generated {len(base_preds)} summaries in {time.perf_counter() - start:.0f} s")
    assert len(base_preds) == N_EVAL
    for i, (pred, ref) in enumerate(zip(base_preds[:5], eval_refs[:5])):
        print(f"\nExample {i}\nMODEL: {pred}\nREFERENCE: {ref}")
    base_rouge = rouge(base_preds, eval_refs)
    print({k: round(v, 1) for k, v in base_rouge.items()},
          f"| mean words: model {words(base_preds):.1f}, reference {words(eval_refs):.1f}")
    print("Exercise 4 checks passed.")
else:
    print("Exercise 4 NF4 baseline skipped: CUDA unavailable.")

Exercise 4 NF4 baseline skipped: CUDA unavailable.


**Question 4.** Read at least five baseline outputs next to their references. (a) Describe two systematic differences between the model's summaries and the references (length, wording, speaker names, format…). (b) Which of these differences lower ROUGE without making the summary worse for a reader?

**(a)** I read the five CPU baseline pairs shown below. Across all 60 examples, the CPU baseline averaged **29.0 words** versus **21.5** for the references. It sometimes restated turns in detail, such as the library's book count and the pool's availability in example 4. Length varied by example. The hotel and interview predictions were shorter than their references. The baseline also often kept anonymized speaker labels where the references used names. It used `#Person2#` for Mary in example 2 and for Adam in example 4, and used speaker labels in place of Lucy and Glenn in example 3.

**(b)** Referring to Mary by her dialogue label and writing “reminisce about their first meeting” instead of “recall the first time they met” preserves the point while changing the word overlap. The cafeteria detail is supported and can help a reader picture the event, even though the reference omits it. In contrast, omitting that Rusty was found in example 3 is a loss of coverage. The CPU diagnostic used Q4_K_M. The five graded NF4 pairs will print above on the T4.

## Exercise 5 · Attach LoRA and train

The adapter configuration was also checked locally using the Qwen configuration on PyTorch's meta device. It targeted 196 layers and produced 18,464,768 trainable parameters. The T4 cell checks the count again after loading NF4 weights.

In [10]:
if HAS_GPU:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import Trainer, TrainingArguments

    checkpointing = {"use_reentrant": False}
    model = prepare_model_for_kbit_training(model, gradient_checkpointing_kwargs=checkpointing)
    lora_config = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    args = TrainingArguments(
        output_dir=str(OUT / "trainer"),
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        fp16=True,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs=checkpointing,
        logging_steps=10,
        logging_nan_inf_filter=False,
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        seed=SEED,
    )
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"trainable parameters: {n_trainable:,} (estimated {N_LORA:,})")
    assert n_trainable == N_LORA
    assert args.fp16 and not args.bf16 and "paged" in str(args.optim)
    assert args.per_device_train_batch_size * args.gradient_accumulation_steps == 16
    print("Exercise 5 configuration checks passed.")
else:
    print("Exercise 5 GPU configuration checks skipped: CUDA unavailable.")

Exercise 5 GPU configuration checks skipped: CUDA unavailable.


In [11]:
def collate(batch):
    width = max(len(row["input_ids"]) for row in batch)
    input_ids = torch.full((len(batch), width), tokenizer.pad_token_id, dtype=torch.long)
    labels = torch.full((len(batch), width), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), width), dtype=torch.long)
    for i, row in enumerate(batch):
        size = len(row["input_ids"])
        input_ids[i, :size] = torch.tensor(row["input_ids"])
        labels[i, :size] = torch.tensor(row["labels"])
        attention_mask[i, :size] = torch.tensor(row["attention_mask"])
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

if torch is not None:
    short = build_example("#Person1#: Hi.", "#Person1# greets someone.")
    padded = collate([short, ex])
    assert padded["labels"][0, len(short["labels"]):].eq(-100).all()
    assert padded["attention_mask"][0, len(short["labels"]):].eq(0).all()
    assert padded["input_ids"][0, len(short["labels"]):].eq(tokenizer.pad_token_id).all()
    print("Exercise 5 collator checks passed.")

if HAS_GPU:
    model.config.use_cache = False
    trainer = Trainer(model=model, args=args, train_dataset=train_data, data_collator=collate)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    trainer.train()
    torch.cuda.synchronize()
    train_minutes = (time.perf_counter() - start) / 60
    peak_gb = torch.cuda.max_memory_allocated() / GB
    model.config.use_cache = True
    print(f"Training time: {train_minutes:.1f} min | peak GPU memory: {peak_gb:.2f} GB "
          f"(before-activation estimate: {estimate['QLoRA']:.2f} GB)")
else:
    print("Exercise 5 training, loss curve, time, and peak memory skipped: CUDA unavailable.")

Exercise 5 collator checks passed.
Exercise 5 training, loss curve, time, and peak memory skipped: CUDA unavailable.


In [12]:
if HAS_GPU:
    import matplotlib.pyplot as plt

    log = [(row["step"], row["loss"]) for row in trainer.state.log_history if "loss" in row]
    steps, losses = zip(*log)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(steps, losses, marker="o")
    ax.set(xlabel="Optimizer step", ylabel="Training loss (summary tokens)", title="QLoRA training loss")
    fig.tight_layout()
    plt.show()
    assert all(math.isfinite(loss) for loss in losses), "A non-finite loss suggests numerical instability."
    assert losses[-1] < losses[0], "Expected the last logged loss to be below the first."
    print("Exercise 5 checks passed.")

**Question 5.** (a) Report your training time and peak memory. Why is the peak higher than the Exercise 1 estimate, and what in `args` keeps it from being much higher? (b) The training loss went down. Why is that **not** enough to conclude that the model summarizes better?

**(a)** Training time and peak GPU allocation require the T4 run. The 1.91 GB calculation excludes activations, logits and temporary buffers. Gradient checkpointing saves fewer intermediate activations and recomputes them in backpropagation. Microbatches of four keep each forward pass small while accumulation gives an effective batch size of 16. FP16 mixed precision and the paged 8-bit optimizer also help control memory use.  
**(b)** Lower teacher-forced training loss means the model better predicts reference tokens on the training set. It can learn reference style or overfit while still omitting key facts or making mistakes on unseen dialogues. Evaluate generated summaries on held-out examples.

## Exercise 6 · Paired validation comparison

In [13]:
def rouge_l_wins(tuned, base, refs):
    if not (len(tuned) == len(base) == len(refs)):
        raise ValueError("The paired comparison needs one prediction per model and reference.")
    return sum(
        scorer.score(ref, after)["rougeL"].fmeasure > scorer.score(ref, before)["rougeL"].fmeasure
        for ref, after, before in zip(refs, tuned, base)
    )

assert rouge_l_wins(["a cat", "a dog"], ["a dog", "a dog"], ["a cat", "a dog"]) == 1
print("Exercise 6 paired scoring check passed (one win and one tie).")

if HAS_GPU:
    start = time.perf_counter()
    tuned_preds = generate_summaries(model, eval_dialogues)
    print(f"Generated {len(tuned_preds)} summaries in {time.perf_counter() - start:.0f} s")
    tuned_rouge = rouge(tuned_preds, eval_refs)
    wins = rouge_l_wins(tuned_preds, base_preds, eval_refs)
    print(f"{'':<12}" + "".join(f"{key:>9}" for key in ROUGE_KEYS) + f"{'words':>8}")
    for name, scores, preds in (("base", base_rouge, base_preds), ("QLoRA", tuned_rouge, tuned_preds)):
        print(f"{name:<12}" + "".join(f"{scores[key]:>9.1f}" for key in ROUGE_KEYS)
              + f"{words(preds):>8.1f}")
    print(f"{'reference':<39}{words(eval_refs):>8.1f}")
    print("Score changes:", {key: round(tuned_rouge[key] - base_rouge[key], 2) for key in ROUGE_KEYS})
    print(f"Mean length change: {words(tuned_preds) - words(base_preds):+.1f} words")
    print(f"QLoRA ROUGE-L higher on {wins} of {N_EVAL} dialogues")
    assert set(tuned_rouge) == set(ROUGE_KEYS) and 0 <= wins <= N_EVAL
    print("Exercise 6 checks passed.")
else:
    print("Exercise 6 tuned generation and comparison skipped: CUDA unavailable.")

Exercise 6 paired scoring check passed (one win and one tie).
Exercise 6 tuned generation and comparison skipped: CUDA unavailable.


**Question 6.** (a) Summarize the change in ROUGE and in length. (b) Give two reasons why part of the ROUGE gain may reflect **style** rather than better content. (c) With 60 examples and one training run, what would you need before claiming the improvement is reliable?

**(a)** The directions and sizes of the changes are unmeasured here. The cell reports the paired ROUGE differences, mean length change, and strict ROUGE-L wins on the T4.  
**(b)** Copying DialogSum's speaker labels and common phrasing raises unigram and bigram overlap without adding facts. Moving closer to reference length can raise F1 by removing valid but unmatched details or by improving recall through a longer summary.  
**(c)** Repeat training across several seeds and evaluate on a larger, locked holdout. Report paired bootstrap confidence intervals for the validation differences and independent blinded ratings to see whether any gain extends to content quality.

## Exercise 7 · Blind rating

For six validation dialogues, score A and B separately on faithfulness and coverage. Use **2** for fully supported or all main points, **1** for a minor factual error or missing main points, and **0** for a major factual error or missing the main point. Read the entire dialogue. Enter four integer scores, a short evidence note, and any invented claim in the ratings cell before executing the reveal.

After entering scores, run only the ratings and reveal cells to use the same generated pairs.

In [14]:
HUMAN_IDS = random.Random(SEED).sample(range(N_EVAL), 6)
if HAS_GPU:
    order = {}
    for i in HUMAN_IDS:
        order[i] = random.Random(SEED + i).choice(["base", "QLoRA"])
        a, b = (base_preds[i], tuned_preds[i]) if order[i] == "base" else (tuned_preds[i], base_preds[i])
        print(f"=================== example {i} ===================")
        print(eval_dialogues[i])
        print(f"\nREFERENCE: {eval_refs[i]}\nA: {a}\nB: {b}\n")
else:
    print("Exercise 7 blind display skipped: the two model predictions are unavailable.")

Exercise 7 blind display skipped: the two model predictions are unavailable.


In [15]:
# Add one entry for each displayed ID before executing the reveal.
# ID: dict(A_faithful=..., A_coverage=..., B_faithful=..., B_coverage=...,
#          note="Evidence from dialogue", invented="A or B and claim, or none"),
ratings = {}

In [16]:
def rating_totals(ratings, order):
    totals = {"base": [0, 0], "QLoRA": [0, 0]}
    for i, row in ratings.items():
        a_model = order[i]
        b_model = "QLoRA" if a_model == "base" else "base"
        for model_name, prefix in ((a_model, "A"), (b_model, "B")):
            for index, field in enumerate(("faithful", "coverage")):
                value = row[f"{prefix}_{field}"]
                assert type(value) is int and 0 <= value <= 2
                totals[model_name][index] += value
    return totals

if HAS_GPU and ratings:
    assert set(ratings) == set(HUMAN_IDS), "Rate all six examples before revealing model names."
    assert all(row["note"].strip() for row in ratings.values()), "Add a note for every example."
    totals = rating_totals(ratings, order)
    disagreements = []
    for i, row in ratings.items():
        a_model = order[i]
        b_model = "QLoRA" if a_model == "base" else "base"
        a, b = (base_preds[i], tuned_preds[i]) if a_model == "base" else (tuned_preds[i], base_preds[i])
        a_rouge = scorer.score(eval_refs[i], a)["rougeL"].fmeasure
        b_rouge = scorer.score(eval_refs[i], b)["rougeL"].fmeasure
        human_delta = row["A_faithful"] + row["A_coverage"] - row["B_faithful"] - row["B_coverage"]
        if (a_rouge - b_rouge) * human_delta < 0:
            disagreements.append(i)
        print(f"Example {i}: A = {a_model} ({a_rouge:.3f}), B = {b_model} ({b_rouge:.3f})")
        print("Evidence:", row["note"], "| Invented claims:", row.get("invented", "unrecorded"))
    for name, (faithfulness, coverage) in totals.items():
        print(f"{name:<6} mean faithfulness {faithfulness / len(ratings):.2f} | "
              f"mean coverage {coverage / len(ratings):.2f} (out of 2)")
    print("Opposite ROUGE-L and summed human preferences:", disagreements)
    print("Exercise 7 checks passed.")
else:
    print("Exercise 7 reveal pending six ratings on actual paired outputs.")

Exercise 7 reveal pending six ratings on actual paired outputs.


**Question 7.** (a) Do your human ratings agree with the ROUGE comparison? Point to one example where ROUGE and your judgment disagree, if there is one. (b) Did either model invent facts? Give an example. (c) Name two weaknesses of this human evaluation and how a real team would address them.

**(a)** Agreement with ROUGE and a concrete disagreement require both sets of generated summaries and the six blind ratings. The reveal lists examples where ROUGE-L and the sum of the two rating scores favor opposite candidates, and preserves the separate faithfulness and coverage scores.  
**(b)** An invented fact requires checking a specific generated claim against the full dialogue. In CPU baseline example 3, the model adds “their friends” to the search. The dialogue only describes Lucy and Glenn walking around the neighborhood. The paired NF4 and QLoRA review is pending. The reveal records the candidate and claim from each rating note.  
**(c)** Six dialogues give a noisy estimate and may miss rare failure modes. Use a larger, stratified sample that includes long and ambiguous cases. One reader's scores can reflect subjective choices. Use independent blinded raters, calibrate the rubric, and report agreement before adjudicating disagreements.

## Exercise 8 · Save the adapter

In [17]:
print(f"Adapter weight estimate at fp32: {N_LORA * 4 / 1e6:.2f} MB")
print(f"Adapter weight estimate at fp16 or bf16: {N_LORA * 2 / 1e6:.2f} MB")
if HAS_GPU:
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
    assert (ADAPTER_DIR / "adapter_model.safetensors").is_file()
    assert (ADAPTER_DIR / "adapter_config.json").is_file()
    for path in sorted(ADAPTER_DIR.iterdir()):
        print(f"{path.name:<32}{path.stat().st_size / 1e6:8.1f} MB")
else:
    print("Exercise 8 adapter save skipped: no trained adapter.")

Adapter weight estimate at fp32: 73.86 MB
Adapter weight estimate at fp16 or bf16: 36.93 MB
Exercise 8 adapter save skipped: no trained adapter.


**Question 8.** The adapter is a small fraction of the 3.1 GB base model. (a) Explain its size from the number of trainable parameters. (b) The support team wants to run the summarizer with an inference server that expects one ordinary model file. Describe the steps, and say what you would re-check afterwards.

**(a)** The adapter contains 18,464,768 learned values. At four bytes per FP32 weight, its tensor payload is about **73.86 MB**, or **36.93 MB** if explicitly saved at 16 bits. There is also a little metadata. The file size printed above after training gives the actual value.  
**(b)** Reload the exact base checkpoint in a supported unquantized dtype. Load the adapter with `PeftModel.from_pretrained`, call `merge_and_unload()`, and save the merged model and tokenizer in the server's supported format. If it requires one weight file, set a shard size larger than the full checkpoint, then verify the generated file and include any configuration the server requires. Re-run held-out prompts to check stop behavior, faithfulness, coverage, ROUGE, latency, and memory. Changing the base precision from NF4 to FP16 or BF16 can change outputs, even after merging the same adapter.

## Reflection

I would keep the model in offline evaluation. On 60 DialogSum validation examples, the CPU baseline scored 29.2 ROUGE-L and averaged 29.0 words against 21.5 for the references. Its lost-dog summary introduced friends who did not appear in the dialogue and omitted that Rusty was found. Those errors matter in a support summary because they change who acted and whether the issue was resolved. The CPU results describe the original model in Q4_K_M. A T4 run and blind ratings are needed to tell whether the adapter improves ROUGE, length, faithfulness, and coverage under the lab's NF4 setup. Higher overlap could still reflect reference-like wording or speaker labels. My next experiment would repeat the paired comparison across several seeds and ask independent reviewers to score a larger held-out sample of authorized support dialogues for factual accuracy, speaker attribution, and coverage of commitments and resolutions.

## Appendix · CPU baseline diagnostic

This optional diagnostic uses the official Qwen **Q4_K_M GGUF** and llama.cpp on CPU, with the same prompts, validation order, greedy decoding, and 96-token cap. Its weights and inference engine differ from the lab's **bitsandbytes NF4** baseline. These results are useful for inspecting baseline style and do not enter the paired comparison.

The Qwen tokenizer and GGUF tokenizer produced identical prompt token IDs for all 60 rows.

In [18]:
cpu_model_path = os.environ.get("QLORA_CPU_GGUF")
if cpu_model_path:
    from llama_cpp import Llama

    cpu_baseline = Llama(
        model_path=cpu_model_path, n_ctx=1024, n_gpu_layers=0,
        n_threads=int(os.environ.get("QLORA_CPU_THREADS", "6")), verbose=False,
    )
    cpu_count = int(os.environ.get("QLORA_CPU_COUNT", str(N_EVAL)))
    assert 5 <= cpu_count <= N_EVAL
    start = time.perf_counter()
    cpu_preds = []
    for dialogue in eval_dialogues[:cpu_count]:
        token_ids = cpu_baseline.tokenize(make_prompt(dialogue).encode(), add_bos=False, special=True)
        completion = cpu_baseline.create_completion(token_ids, max_tokens=96, temperature=0)
        cpu_preds.append(completion["choices"][0]["text"].strip())
    cpu_refs = eval_refs[:cpu_count]
    print(f"CPU Q4_K_M baseline: {len(cpu_preds)} summaries in {time.perf_counter() - start:.1f} s")
    for i, (pred, ref) in enumerate(zip(cpu_preds[:5], cpu_refs[:5])):
        print(f"\nExample {i}\nMODEL: {pred}\nREFERENCE: {ref}")
    print({k: round(v, 1) for k, v in rouge(cpu_preds, cpu_refs).items()})
    print(f"Mean words: model {words(cpu_preds):.1f}, reference {words(cpu_refs):.1f}")
else:
    print("Optional CPU diagnostic requires QLORA_CPU_GGUF pointing to the Q4_K_M file.")

CPU Q4_K_M baseline: 60 summaries in 149.2 s

Example 0
MODEL: #Person2# enjoyed their stay at the Highland Hotel, with good facilities, reasonable prices, and excellent service, but had a problem with internet access.
REFERENCE: #Person2# enjoys #Person2#'s weekend at the highland hotel because of the hotel's excellent and reasonably priced restaurant and good service. #Person2# introduces the hotel's facilities, weekend discount, and its interesting tip policy and suggests #Person1# make a reservation in advance.

Example 1
MODEL: Peter Wilson works for Green Peace, an anti-nuclear organization that arranges protests and harasses ships dumping radioactive waste in the Atlantic Ocean.
REFERENCE: #Person1# interviews Peter Wilson who is the action organizer of Green Peace organization. #Person1# asks Peter to introduce to the audience what Green Peace is and what work it does. Peter also introduces detailed anti-nuclear campaigns.

Example 2
MODEL: #Person1# and #Person2# reminisce abo